In [15]:
import pandas as pd
import numpy as np
import time
import joblib
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import roc_auc_score, f1_score, classification_report

In [16]:
# --- FASE 1: Carga y corrección inicial ---
def cargar_y_corregir_datos(ruta_train, ruta_test, col_anomala=None, valor_anomalo=None):
    """Carga los datos y corrige valores centinela reemplazándolos por nulos."""
    print("=" * 60)
    print("📥 FASE 1: CARGA DE DATOS Y CORRECCIÓN DE ANOMALÍAS")
    print("=" * 60)
    
    train = pd.read_csv(ruta_train)
    test = pd.read_csv(ruta_test)
    
    if col_anomala and valor_anomalo is not None:
        # Reemplazar valor centinela por nulos[cite: 9]
        train[col_anomala] = train[col_anomala].replace(valor_anomalo, np.nan)
        test[col_anomala] = test[col_anomala].replace(valor_anomalo, np.nan)
        print(f"✔️ Anomalía corregida en la columna '{col_anomala}'.")
        
    print(f"Tamaño Train: {train.shape} | Tamaño Test: {test.shape}")
    return train, test

# --- FASE 2: Unión y limpieza de nulos excesivos ---
def limpiar_nulos_excesivos(train, test, umbral_obs=0.5, umbral_attr=0.6, auto_eliminar=True):
    """Une los datasets y elimina filas/columnas con demasiados nulos sin requerir input humano."""
    print("\n" + "=" * 60)
    print("🧹 FASE 2: LIMPIEZA DE NULOS EXCESIVOS")
    print("=" * 60)
    
    train_copy = train.copy()
    test_copy = test.copy()
    test_copy['TARGET'] = 2 # Etiqueta temporal[cite: 9]
    
    df = pd.concat([train_copy, test_copy], axis=0, ignore_index=True)
    
    predictoras = df.drop(columns=['TARGET'])
    n_obs, n_attr = predictoras.shape
    
    # 1. Limpieza de observaciones (Filas)[cite: 9]
    obs_muchos_vacios = (predictoras.isnull().sum(axis=1) / n_attr) >= umbral_obs
    if obs_muchos_vacios.sum() > 0 and auto_eliminar:
        df = df.loc[~obs_muchos_vacios]
        print(f"✔️ Se eliminaron {obs_muchos_vacios.sum()} filas por exceso de nulos.")
        
    # 2. Limpieza de atributos (Columnas)[cite: 9]
    predictoras = df.drop(columns=['TARGET'])
    n_obs, n_attr = predictoras.shape
    attr_muchos_vacios = (predictoras.isnull().sum(axis=0) / n_obs) >= umbral_attr
    if attr_muchos_vacios.sum() > 0 and auto_eliminar:
        cols_a_mantener = predictoras.columns[~attr_muchos_vacios].tolist() + ['TARGET']
        df = df[cols_a_mantener]
        print(f"✔️ Se eliminaron {attr_muchos_vacios.sum()} columnas por exceso de nulos.")

    print(f"Dimensiones tras limpieza: {df.shape}")
    return df

# --- FASE 3: Preprocesamiento dinámico ---
def preprocesar_datos(df_unido, cols_enteras_conocidas=None):
    """Imputa, escala y da formato a las variables. Totalmente agnóstico al dataset."""
    print("\n" + "=" * 60)
    print("⚙️ FASE 3: PREPROCESAMIENTO (Imputación, Escalado y Categorización)")
    print("=" * 60)
    
    df = df_unido.copy()
    
    # Si no se provee lista de enteras, usamos una vacía
    if cols_enteras_conocidas is None:
        cols_enteras_conocidas = []
        
    # Filtramos para asegurar que las columnas enteras realmente existan en este dataset
    cols_enteras = [col for col in cols_enteras_conocidas if col in df.columns]
    
    # Separar numéricas y categóricas/texto dinámicamente[cite: 9]
    cols_numericas = df.select_dtypes(include=['number']).columns.tolist()
    if 'TARGET' in cols_numericas:
        cols_numericas.remove('TARGET')
        
    cols_continuas = [col for col in cols_numericas if col not in cols_enteras]
    cols_categoricas = df.select_dtypes(exclude=['number']).columns.tolist() # Captura object, str, bool, etc.[cite: 9]
    
    # A. Imputación y Formato de Categóricas[cite: 9]
    if cols_categoricas:
        imputer_cat = SimpleImputer(strategy='constant', fill_value='Desconocido')
        df[cols_categoricas] = imputer_cat.fit_transform(df[cols_categoricas])
        for col in cols_categoricas:
            df[col] = df[col].astype('category')
        print(f"✔️ {len(cols_categoricas)} variables categóricas imputadas y convertidas a 'category'.")

    # B. Imputación y Escalado de Numéricas Continuas[cite: 9]
    if cols_continuas:
        imputer_num = SimpleImputer(strategy='median')
        df[cols_continuas] = imputer_num.fit_transform(df[cols_continuas])
        
        scaler = RobustScaler()
        df[cols_continuas] = scaler.fit_transform(df[cols_continuas])
        print(f"✔️ {len(cols_continuas)} variables continuas imputadas (mediana) y escaladas (RobustScaler).")

    # C. Imputación de Numéricas Enteras[cite: 9]
    if cols_enteras:
        imputer_int = SimpleImputer(strategy='median')
        df[cols_enteras] = imputer_int.fit_transform(df[cols_enteras])
        df[cols_enteras] = df[cols_enteras].round().astype('Int64')
        print(f"✔️ {len(cols_enteras)} variables discretas imputadas y forzadas a entero.")

    return df

# --- FASE 4: Separación de Sets ---
def preparar_sets_entrenamiento(df_procesado, test_size=0.20, random_state=42):
    """Separa el dataset de nuevo en train, validación y test final."""
    print("\n" + "=" * 60)
    print("✂️ FASE 4: SEPARACIÓN DE SETS DE DATOS")
    print("=" * 60)
    
    train_limpio = df_procesado[df_procesado['TARGET'].isin([0, 1])].copy()
    test_limpio = df_procesado[df_procesado['TARGET'] == 2].copy()

    X = train_limpio.drop(columns=['TARGET'])
    y = train_limpio['TARGET'].astype(int)
    X_test_final = test_limpio.drop(columns=['TARGET'])

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    
    print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test Final: {X_test_final.shape}")
    return X_train, X_val, y_train, y_val, X_test_final

def entrenar_lightgbm(X_train, y_train, X_val, y_val, learning_rate=0.05, num_leaves=31, num_boost_round=1000):
    
    # Configurar los parámetros del modelo incluyendo is_unbalance
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'learning_rate': learning_rate,
        'num_leaves': num_leaves,
        'n_jobs': -1,  # OpenMP activado
        'is_unbalance': True, # <--- ¡CLAVE! Activa el balanceo interno de LightGBM
        'verbose': -1
    }

    train_data = lgb.Dataset(X_train, label=y_train)
    val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

    callbacks = [
        lgb.early_stopping(stopping_rounds=50, verbose=False), 
        lgb.log_evaluation(period=0)
    ]

    modelo_lgb = lgb.train(
        params,
        train_data,
        num_boost_round=num_boost_round,
        valid_sets=[train_data, val_data],
        callbacks=callbacks
    )

    y_pred_prob = modelo_lgb.predict(X_val)
    y_pred_class = (y_pred_prob > 0.5).astype(int)

    auc = roc_auc_score(y_val, y_pred_prob)
    f1 = f1_score(y_val, y_pred_class)
    return modelo_lgb, {'roc_auc': auc, 'f1_score': f1}, y_pred_class

def entrenar_xgboost(X_train, y_train, X_val, y_val, learning_rate=0.05, max_depth=6, num_boost_round=1000, scale_weight=1.0):
    
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',           
        'tree_method': 'hist',          
        'learning_rate': learning_rate,
        'max_depth': max_depth,
        'n_jobs': -1,  # OpenMP activado
        'scale_pos_weight': scale_weight, # <--- ¡CLAVE! Le pasamos el peso calculado
        'random_state': 42
    }

    dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
    dval = xgb.DMatrix(X_val, label=y_val, enable_categorical=True)

    evals = [(dtrain, 'entrenamiento'), (dval, 'validacion')]

    modelo_xgb = xgb.train(
        params,
        dtrain,
        num_boost_round=num_boost_round,
        evals=evals,
        early_stopping_rounds=50,
        verbose_eval=False
    )

    y_pred_prob = modelo_xgb.predict(dval)
    y_pred_class = (y_pred_prob > 0.5).astype(int)

    auc = roc_auc_score(y_val, y_pred_prob)
    f1 = f1_score(y_val, y_pred_class)
    return modelo_xgb, {'roc_auc': auc, 'f1_score': f1}, y_pred_class

In [17]:
# ==========================================
# 🚀 BLOQUE MAIN: EJECUCIÓN DEL PIPELINE
# ==========================================

# 1. Parámetros específicos del Dataset actual (Se cambian según el proyecto)
RUTA_TRAIN = 'application_train.csv'
RUTA_TEST = 'application_test.csv'
COLUMNA_ANOMALA = 'DAYS_EMPLOYED'
VALOR_ANOMALO = 365243

COLUMNAS_QUE_DEBEN_SER_ENTERAS = [
    'CNT_FAM_MEMBERS', 'OBS_30_CNT_SOCIAL_CIRCLE', 'DEF_30_CNT_SOCIAL_CIRCLE', 
    'OBS_60_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE', 'DAYS_LAST_PHONE_CHANGE',
    'AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_DAY', 'AMT_REQ_CREDIT_BUREAU_WEEK', 
    'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR',
    'DAYS_EMPLOYED'
]

# 2. Ejecución Secuencial del Pipeline de Datos
df_train_raw, df_test_raw = cargar_y_corregir_datos(
    RUTA_TRAIN, RUTA_TEST, COLUMNA_ANOMALA, VALOR_ANOMALO
)

df_unificado = limpiar_nulos_excesivos(df_train_raw, df_test_raw, auto_eliminar=True)

df_listo = preprocesar_datos(df_unificado, cols_enteras_conocidas=COLUMNAS_QUE_DEBEN_SER_ENTERAS)

X_train, X_val, y_train, y_val, X_test_final = preparar_sets_entrenamiento(df_listo)

# 3. Preparativos para MLOps (Paralelismo de Modelos)
peso_clase_positiva = sum(y_train == 0) / sum(y_train == 1)

print("\n" + "=" * 60)
print("🧠 FASE 5: ENTRENAMIENTO PARALELO DE MODELOS")
print("=" * 60)
start_time = time.time()

resultados = joblib.Parallel(n_jobs=2, backend="threading")([
    joblib.delayed(entrenar_lightgbm)(X_train, y_train, X_val, y_val),
    joblib.delayed(entrenar_xgboost)(X_train, y_train, X_val, y_val, scale_weight=peso_clase_positiva)
])

(modelo_lgb, metricas_lgb, preds_lgb) = resultados[0]
(modelo_xgb, metricas_xgb, preds_xgb) = resultados[1]

print(f"⏱️ Tiempo total de entrenamiento concurrente: {time.time() - start_time:.2f} segundos")

📥 FASE 1: CARGA DE DATOS Y CORRECCIÓN DE ANOMALÍAS
✔️ Anomalía corregida en la columna 'DAYS_EMPLOYED'.
Tamaño Train: (307511, 122) | Tamaño Test: (48744, 121)

🧹 FASE 2: LIMPIEZA DE NULOS EXCESIVOS
✔️ Se eliminaron 14 filas por exceso de nulos.
✔️ Se eliminaron 17 columnas por exceso de nulos.
Dimensiones tras limpieza: (356241, 105)

⚙️ FASE 3: PREPROCESAMIENTO (Imputación, Escalado y Categorización)
✔️ 15 variables categóricas imputadas y convertidas a 'category'.
✔️ 76 variables continuas imputadas (mediana) y escaladas (RobustScaler).
✔️ 13 variables discretas imputadas y forzadas a entero.

✂️ FASE 4: SEPARACIÓN DE SETS DE DATOS
Train: (245998, 104) | Val: (61500, 104) | Test Final: (48743, 104)

🧠 FASE 5: ENTRENAMIENTO PARALELO DE MODELOS
⏱️ Tiempo total de entrenamiento concurrente: 31.37 segundos
